# Phase 1: Data Acquisition
This notebook queries the NASA Exoplanet Archive to build a balanced training/testing labels list of TESS targets (confirmed planets, eclipsing binaries, and false positives), downloads their raw light curve data from MAST, and saves them locally.

This notebook is intended for early-stage ML learners (2nd-year engineering students) to understand and interact with astrophysical data archives.

In [ ]:
import sys
import os

# Add the parent directory to Python search path so we can import local src modules
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
import glob
from src.data_loader import fetch_labeled_list, download_light_curves, load_light_curve

## 1. Fetch Labeled Stars
We query the NASA Exoplanet Archive TOI database using `astroquery` for 10 confirmed planets, 10 eclipsing binaries, and 10 false positives (30 stars total).

In [ ]:
labels_df = fetch_labeled_list(num_per_class=10)
labels_df.to_csv('../data/labels.csv', index=False)
labels_df.head(15)

## 2. Download Light Curve Data
For each star, we download its TESS light curve data using `lightkurve` from the MAST archive. We save each star's time and flux data to a CSV in `data/raw/`.

In [ ]:
download_light_curves(
    labels_df=labels_df, 
    output_dir='../data/raw', 
    failures_path='../outputs/download_failures.csv'
)

## 3. Verify Downloaded Data
Let's load one of the raw light curves to inspect its format.

In [ ]:
raw_files = glob.glob('../data/raw/*.csv')
if raw_files:
    sample_file = raw_files[0]
    print(f"Loading sample file: {sample_file}")
    sample_lc = load_light_curve(sample_file)
    print(sample_lc.head())
else:
    print("No raw files found yet.")